In [3]:
import torch
import torchvision
import torch.nn.functional as F
from alexnet import alexnet
from torchvision import transforms
from torch.utils.data.dataloader import DataLoader

# device
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = 'cpu'

# Reproducibility
torch.manual_seed(123)
if device=='cuda':
    torch.cuda.manual_seed_all(123)

# Setup the transformation
trans1 = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.ToTensor(), # cast to Tensor type
    transforms.Normalize((0.1307,), (0.3081,)) # normalize the image with mean, std value
])

# Setup the transformation
trans2 = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),  # cast to Tensortype
    transforms.Normalize((0.1307,), (0.3081,))  # normalize the image with mean, std value
])

# Load the image set
# Download the train and test dataset, respectively
# train image set(60,000, 28, 28), test image set(10,000, 28, 28)
# To train alexnet, upscale mnist to size (256, 256) after applying random cropping to size (224, 224)
train_X = torchvision.datasets.MNIST(root='data', train=True, transform=trans1, download=True)
test_X = torchvision.datasets.MNIST(root='data', train=False, transform=trans2, download=True)

# Setup the loader
# for train data, data random shuffle and drop_last are used.
# for test data, data random shuffle and drop_last are not used for same comparison
# pin_memory option can enhance the GPU utility
train_loader = DataLoader(train_X, batch_size=32, shuffle=True, drop_last=True, pin_memory=True)
test_loader = DataLoader(test_X, batch_size=100, shuffle=False, drop_last=False, pin_memory=True)

In [4]:
# Setup the model
model = alexnet(num_classes=10).to(device)
print(model) # print the defined model
# Setup the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

AlexNet(
  (features): Sequential(
    (0): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

In [5]:
# Training
# 15 epoch: each epoch iterate N / batch size (N: the number of data)
for epoch in range(15):
    for idx, (images, labels) in enumerate(train_loader):
        images, labels = images.float().to(device), labels.long().to(device)
        # Extract output of single layer
        # Output dimension will be (M, K)
        # M: batch size, K: the number of class
        hypothesis = model(images)
        # Calculate cross-entropy loss
        # Input shape: [M,K], target shape: [M,]
        # Label data is scalar value of class (0~9)
        # Detach() can prevent the backpropagation for label data
        cost = F.cross_entropy(
            input=hypothesis,
            target=labels.detach()
        )
        # Gradient initialization
        optimizer.zero_grad()
        # Calculate gradient
        cost.backward()
        # Update parameters
        optimizer.step()
        # Calculate accuracy
        # 0: column-wise, 1:row-wise
        prob = hypothesis.softmax(dim=1)
        # Extract index of maximum possibility component
        pred = prob.argmax(dim=1)
        # Compare the equality between prediction and ground-truth
        acc = pred.eq(labels).float().mean()
        # Display at every 128 iteration
        if (idx+1) % 128 == 0:
            print(f'Train-Iteration: {idx+1}, Loss: {cost.item()}, Accuracy: {acc.item()}')

Train-Iteration: 128, Loss: 0.8386469483375549, Accuracy: 0.6875
Train-Iteration: 256, Loss: 0.37764301896095276, Accuracy: 0.875
Train-Iteration: 384, Loss: 0.2296328991651535, Accuracy: 0.9375
Train-Iteration: 512, Loss: 0.25607621669769287, Accuracy: 0.875
Train-Iteration: 640, Loss: 0.1305437684059143, Accuracy: 0.96875
Train-Iteration: 768, Loss: 0.04174506664276123, Accuracy: 1.0
Train-Iteration: 896, Loss: 0.06656541675329208, Accuracy: 1.0
Train-Iteration: 1024, Loss: 0.20177310705184937, Accuracy: 0.96875
Train-Iteration: 1152, Loss: 0.2998546361923218, Accuracy: 0.90625
Train-Iteration: 1280, Loss: 0.09825839102268219, Accuracy: 0.96875
Train-Iteration: 1408, Loss: 0.26745355129241943, Accuracy: 0.90625
Train-Iteration: 1536, Loss: 0.08338853716850281, Accuracy: 0.96875
Train-Iteration: 1664, Loss: 0.132629856467247, Accuracy: 0.9375
Train-Iteration: 1792, Loss: 0.18468844890594482, Accuracy: 0.96875
Train-Iteration: 128, Loss: 0.06350007653236389, Accuracy: 0.96875
Train-Ite

In [6]:
# Evaluation
# The parameters must be fixed during the inference
with torch.no_grad():
    for idx, (images, labels) in enumerate(test_loader):
        images, labels = images.float().to(device), labels.long().to(device)
        # Extract output of model
        hypothesis = model(images)
        # Calculate cross-entropy loss
        cost = F.cross_entropy(
            input=hypothesis,
            target=labels)
        # Calculate accuracy
        # 0: column-wise, 1: row-wise
        prob = hypothesis.softmax(dim=1)
        # Extract index of maximum possibility component
        pred = prob.argmax(dim=1)
        # Compare the equality between prediction and ground-truth
        acc += pred.eq(labels).float().mean()
    print(f'Test-Accuracy: {acc/len(test_loader)}')

Test-Accuracy: 0.9959748387336731
